# Caracal Bench s01 - TPU one-shot

Mede checkpoint `caracal-base-3b-s01` vs Qwen base em MMLU computer_security (~100 MCQ).

**Antes de Run All:**
1. Settings -> Accelerator -> TPU VM v3-8 (ou v5e-8)
2. Settings -> Internet -> ON
3. Settings -> Persistence -> Variables and Files

Probe (51 prompts) + HumanEval (164) usam `.generate()` -> XLA recompila por shape -> rodar em GPU T4 separadamente.

In [ ]:
CHECKPOINT_DATASET = "pedroafonso2/caracal-base-3b-s01"
OUTPUT_DATASET = "pedroafonso2/caracal-bench-s01"
EXTRA_FLAGS = ["--skip-probe", "--skip-humaneval"]
print(f"checkpoint={CHECKPOINT_DATASET} -> TPU bench (MMLU only)")

In [ ]:
!pip install -q 'transformers>=4.46.0' 'peft>=0.13.0' 'datasets>=3.0.0' kaggle

In [ ]:
import os
import subprocess

if not os.path.exists("/kaggle/working/caracal-1"):
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "-b",
            "dev",
            "https://github.com/iterate-labs-ai/caracal-1.git",
            "/kaggle/working/caracal-1",
        ],
        check=True,
    )
os.chdir("/kaggle/working/caracal-1")
rev = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
print(f"cloned, HEAD={rev}", flush=True)

In [ ]:
import subprocess

ckpt_dir = "/kaggle/working/ckpt-s01"
subprocess.run(
    [
        "kaggle",
        "datasets",
        "download",
        "-d",
        CHECKPOINT_DATASET,
        "-p",
        ckpt_dir,
        "--unzip",
        "--force",
    ],
    check=True,
)
subprocess.run(["ls", "-la", ckpt_dir], check=True)

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-u",
    "eval/run_bench_all.py",
    "--adapter",
    ckpt_dir,
    "--out-dir",
    "/kaggle/working/bench-s01",
] + EXTRA_FLAGS
print("Running adapter bench:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-u",
    "eval/run_bench_all.py",
    "--out-dir",
    "/kaggle/working/bench-base",
] + EXTRA_FLAGS
print("Running base bench:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

In [ ]:
import json
from pathlib import Path

adapter_sum = json.loads(Path("/kaggle/working/bench-s01/summary.json").read_text())
base_sum = json.loads(Path("/kaggle/working/bench-base/summary.json").read_text())


def metric(s, name, key):
    step = s.get(name, {})
    if step.get("status") != "ok":
        return None
    return step.get("metrics", {}).get(key)


rows = [
    ("probe.mean_ppl", "probe", "mean_ppl"),
    ("probe.cwe_hit_rate", "probe", "cwe_hit_rate"),
    ("mmlu_security.acc", "mmlu_security", "accuracy"),
    ("humaneval.pass_at_1", "humaneval", "pass_at_1"),
]
deltas = {}
for label, name, key in rows:
    a = metric(adapter_sum, name, key)
    b = metric(base_sum, name, key)
    deltas[label] = {
        "adapter": a,
        "base": b,
        "delta": (a - b) if (a is not None and b is not None) else None,
    }

diff = {
    "checkpoint_dataset": CHECKPOINT_DATASET,
    "adapter": adapter_sum,
    "base": base_sum,
    "deltas": deltas,
}
pub_dir = Path("/kaggle/working/bench-published")
pub_dir.mkdir(parents=True, exist_ok=True)
(pub_dir / "bench_s01_vs_base.json").write_text(json.dumps(diff, indent=2))
print(json.dumps(deltas, indent=2))

In [ ]:
import json
import subprocess

metadata = {
    "title": "Caracal Bench s01 vs Base",
    "id": OUTPUT_DATASET,
    "licenses": [{"name": "Apache-2.0"}],
}
(pub_dir / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))

r = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(pub_dir), "--public"],
    capture_output=True,
    text=True,
    check=False,
)
print(r.stdout, r.stderr)
if r.returncode != 0:
    subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(pub_dir), "-m", "bench s01"], check=True
    )
print(f"Published -> {OUTPUT_DATASET}")

## Ler depois

```bash
kaggle datasets download -d pedroafonso2/caracal-bench-s01 -p . --unzip
cat bench_s01_vs_base.json | jq .deltas
```